# 01 — Data Exploration

First-pass audit of every CSV in `data/raw/`. We answer four questions per file before we trust anything downstream:

1. **Shape** — rows × columns. Catches truncated downloads.
2. **Schema** — dtypes (especially: did dates parse?).
3. **Nulls** — where, how much.
4. **Date range / coverage** — what time window does the data actually cover?

Re-run this notebook any time `python -m src.load_data` pulls fresh data.

## 1. Setup

Add the project root to `sys.path` so the `src` package imports cleanly when this notebook is opened on its own.

In [1]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import pandas as pd
from IPython.display import display, Markdown

from src.load_data import load_all, RAW_DIR, EXPECTED_FILES
from src.plotting import save_table

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 200)
print("Reading from:", RAW_DIR)

Reading from: C:\Github\GLP1_Analysis\data\raw


## 2. Are all expected files present?

Cheap sanity check before we read anything. If a file is missing, run `python -m src.load_data` from the project root to grab a fresh copy from Kaggle.

In [2]:
presence = pd.DataFrame(
    [{"file": f, "present": (RAW_DIR / f).exists()} for f in EXPECTED_FILES]
)
display(presence)
missing = presence.loc[~presence["present"], "file"].tolist()
assert not missing, f"Missing files in data/raw/: {missing} — run `python -m src.load_data`."

,file,present
0,drugs_overview.csv,True
1,adverse_events.csv,True
2,adverse_events_summary.csv,True
3,clinical_trials.csv,True
4,stock_prices.csv,True
5,search_trends.csv,True
6,wikipedia_summaries.csv,True
7,data_dictionary.csv,True


## 3. Load every CSV at once

`load_all()` returns a dict keyed by short name. Loaders apply the right `parse_dates` and coerce the seriousness flags to real booleans.

In [3]:
data = load_all()
for name, df in data.items():
    print(f"{name:>12}: {df.shape[0]:>7,} rows x {df.shape[1]:>2} cols")

       drugs:      10 rows x  7 cols
          ae: 149,209 rows x 16 cols
  ae_summary:  11,093 rows x  8 cols
      trials:   1,953 rows x 11 cols
      stocks:   4,680 rows x  9 cols
      trends:   3,500 rows x  4 cols
        wiki:       9 rows x  4 cols
  dictionary:      50 rows x  4 cols


## 4. Per-file audit

For each DataFrame we print the schema with null counts, then show 3 sample rows. Pattern is identical for every file so the loop stays short.

In [4]:
def schema(df: pd.DataFrame) -> pd.DataFrame:
    """One row per column: dtype, null count, % null, unique count."""
    return pd.DataFrame({
        "dtype": df.dtypes.astype(str),
        "nulls": df.isna().sum(),
        "null_pct": (df.isna().mean() * 100).round(2),
        "n_unique": df.nunique(dropna=True),
    })

for name, df in data.items():
    display(Markdown(f"### `{name}` &mdash; {df.shape[0]:,} rows × {df.shape[1]} cols"))
    display(schema(df))
    display(df.head(3))

### `drugs` &mdash; 10 rows × 7 cols

,dtype,nulls,null_pct,n_unique
generic_name,object,0,0.0,10
brand_names,object,3,30.0,7
manufacturer,object,0,0.0,5
indication,object,0,0.0,4
fda_first_approval_date,datetime64[ns],3,30.0,7
is_investigational,bool,0,0.0,2
drug_class,object,0,0.0,6


,generic_name,brand_names,manufacturer,indication,fda_first_approval_date,is_investigational,drug_class
0,semaglutide,OZEMPIC; WEGOVY; RYBELSUS,Novo Nordisk,type-2-diabetes/obesity,2017-12-05,False,GLP-1 receptor agonist
1,tirzepatide,MOUNJARO; ZEPBOUND,Eli Lilly,type-2-diabetes/obesity,2022-05-13,False,dual GIP/GLP-1 receptor agonist
2,liraglutide,VICTOZA; SAXENDA,Novo Nordisk,type-2-diabetes/obesity,2010-01-25,False,GLP-1 receptor agonist


### `ae` &mdash; 149,209 rows × 16 cols

,dtype,nulls,null_pct,n_unique
safetyreportid,int64,0,0.00,54170
generic_name,object,0,0.00,7
brand_queried,object,0,0.00,12
receive_date,datetime64[ns],0,0.00,2947
country,object,0,0.00,73
serious,bool,0,0.00,2
seriousness_death,bool,0,0.00,2
seriousness_lifethreatening,bool,0,0.00,2
seriousness_hospitalization,bool,0,0.00,2
seriousness_disabling,bool,0,0.00,2


...

,dtype,nulls,null_pct,n_unique
generic_name,object,0,0.00,9
wiki_title,object,0,0.00,9
summary,object,0,0.00,9
image_url,object,2,22.22,7


,generic_name,wiki_title,summary,image_url
0,semaglutide,Semaglutide,Semaglutide is an anti-diabetic medication use...,https://upload.wikimedia.org/wikipedia/commons...
1,tirzepatide,Tirzepatide,Tirzepatide is a gastric inhibitory polypeptid...,https://upload.wikimedia.org/wikipedia/commons...
2,liraglutide,Liraglutide,"Liraglutide, sold under the brand name Victoza...",https://upload.wikimedia.org/wikipedia/commons...


### `dictionary` &mdash; 50 rows × 4 cols

,dtype,nulls,null_pct,n_unique
file,object,0,0.0,7
column,object,0,0.0,48
type,object,0,0.0,6
description,object,0,0.0,50


,file,column,type,description
0,drugs_overview.csv,generic_name,string,Generic (INN) drug name
1,drugs_overview.csv,brand_names,string,"Brand names, '; ' separated"
2,drugs_overview.csv,manufacturer,string,Primary manufacturer


## 5. Date ranges

Quick check that each time-series covers the window we expect (2017-2026-ish). Surprises here usually mean a parse error upstream.

In [5]:
date_cols = {
    "ae":     "receive_date",
    "trials": "start_date",
    "stocks": "date",
    "trends": "date",
}
rows = []
for key, col in date_cols.items():
    s = data[key][col]
    rows.append({"dataset": key, "col": col, "min": s.min(), "max": s.max(), "n_nulls": s.isna().sum()})
date_summary = pd.DataFrame(rows)
display(date_summary)

,dataset,col,min,max,n_nulls
0,ae,receive_date,2012-11-23,2025-01-17,0
1,trials,start_date,2004-05-28,2027-09-01,637
2,stocks,date,2017-01-03,2026-04-24,0
3,trends,date,2018-01-01,2026-04-01,0


## 6. Coverage by drug

Different datasets cover different drug subsets. Important for Q6 (investigational drugs) — if CagriSema isn't in `clinical_trials`, no amount of filtering will conjure it up.

In [6]:
ae_drugs     = set(data["ae"]["generic_name"].dropna().unique())
trial_drugs  = set(data["trials"]["drug_query"].dropna().unique())
ref_drugs    = set(data["drugs"]["generic_name"].dropna().unique())
all_drugs    = sorted(ref_drugs | ae_drugs | trial_drugs)

coverage = pd.DataFrame({
    "drug": all_drugs,
    "in_drugs_overview": [d in ref_drugs for d in all_drugs],
    "in_adverse_events": [d in ae_drugs for d in all_drugs],
    "in_clinical_trials": [d in trial_drugs for d in all_drugs],
})
display(coverage)

,drug,in_drugs_overview,in_adverse_events,in_clinical_trials
0,albiglutide,True,True,True
1,cagrilintide-semaglutide,True,False,False
2,dulaglutide,True,True,True
3,exenatide,True,True,True
4,liraglutide,True,True,True
5,lixisenatide,True,True,True
6,orforglipron,True,False,True
7,retatrutide,True,False,True
8,semaglutide,True,True,True
9,tirzepatide,True,True,True


## 7. Quick volume views

How are reports / trials / search interest distributed at first glance? Useful to spot any drug with a sample so small it'll need a footnote later.

In [7]:
display(Markdown("**FAERS reports per drug (reaction-level rows):**"))
display(data["ae"].groupby("generic_name").size().sort_values(ascending=False).rename("n_rows").to_frame())

display(Markdown("**Clinical trials per drug:**"))
display(data["trials"].groupby("drug_query").size().sort_values(ascending=False).rename("n_trials").to_frame())

display(Markdown("**Search trends — geos and terms:**"))
display(data["trends"][["geo", "term"]].drop_duplicates().sort_values(["geo", "term"]).reset_index(drop=True))

**FAERS reports per drug (reaction-level rows):**

,n_rows
generic_name,
semaglutide,45818
exenatide,30281
liraglutide,26447
tirzepatide,21856
albiglutide,12836
dulaglutide,11914
lixisenatide,57


**Clinical trials per drug:**

,n_trials
drug_query,
semaglutide,698
liraglutide,483
exenatide,348
tirzepatide,174
dulaglutide,93
lixisenatide,59
orforglipron,44
albiglutide,28
retatrutide,26


**Search trends — geos and terms:**

,geo,term
0,AE,GLP-1
1,AE,Mounjaro
2,AE,Ozempic
3,AE,Wegovy
4,AE,Zepbound
5,GB,GLP-1
6,GB,Mounjaro
7,GB,Ozempic
8,GB,Wegovy
9,GB,Zepbound


## 8. Persist an overview to disk

Drops a one-line-per-file CSV into `outputs/tables/` so the audit can be shared without re-running the notebook.

In [8]:
overview = pd.DataFrame([
    {
        "dataset": name,
        "rows": df.shape[0],
        "cols": df.shape[1],
        "total_nulls": int(df.isna().sum().sum()),
        "cols_with_nulls": int((df.isna().sum() > 0).sum()),
    }
    for name, df in data.items()
]).sort_values("rows", ascending=False).reset_index(drop=True)

out = save_table(overview, "01_dataset_overview")
print(f"Saved overview to: {out}")
display(overview)

Saved overview to: C:\Github\GLP1_Analysis\outputs\tables\01_dataset_overview.csv


,dataset,rows,cols,total_nulls,cols_with_nulls
0,ae,149209,16,159310,2
1,ae_summary,11093,8,2278,1
2,stocks,4680,9,0,0
3,trends,3500,4,0,0
4,trials,1953,11,1816,3
5,dictionary,50,4,0,0
6,drugs,10,7,6,2
7,wiki,9,4,2,1


## Key Findings (exploration)

- **8 CSVs** loaded; no missing files; total ~165k rows of useful data.
- **Adverse events** are the heaviest table (~149k reaction-level rows, ~54k unique reports after dedup). Covers **2012-11 → 2025-01**, slightly wider than the dataset's advertised 2017-2026.
- **7 approved GLP-1 drugs** in FAERS (semaglutide, tirzepatide, liraglutide, dulaglutide, exenatide, lixisenatide, albiglutide). **No investigational drugs in AE** — investigational drugs only appear in `clinical_trials.csv`.
- **Clinical trials** cover **9 of the 10** drugs in `drugs_overview` — CagriSema (`cagrilintide-semaglutide`) has **zero trials in the file**, which will limit Q6.
- **Search trends** is much narrower than expected: only **7 geos** (US, GB, IN, PK, SA, AE, WORLD) and **5 terms** (GLP-1, Ozempic, Wegovy, Mounjaro, Zepbound). Geographic questions will need to be scoped to that footprint.
- **`country='UNK'`** is the second-largest country bucket in FAERS (~6k reports) — must be excluded from geo plots, not treated as a country.
- Date coverage is good and aligned with the question set; the `seriousness_hospitalization` flag is already a clean boolean (no string `"1"` / NaN mix to clean up).